# Introducción

El dataset consta de aproximadamente 300 imágenes en las cuales cada imagen puede contener múltiples *bounding boxes* (cajas delimitadoras). El alcance de esta prueba de concepto es demostrar de manera empírica y teórica por qué las herramientas tradicionales de Machine Learning de propósito general, específicamente `scikit-learn`, no son viables para resolver tareas de Detección de Objetos (Object Detection).

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier

## Pérdida de Contexto Espacial

Para procesar una imagen 2D con un modelo lineal o un perceptrón multicapa (MLP) tradicional como el `MLPClassifier` de scikit-learn, la matriz de píxeles debe ser aplanada (flattened) en un vector unidimensional (1D). Este proceso destruye de manera inherente las relaciones topológicas y espaciales entre los píxeles vecinos.

En contraste, las Redes Neuronales Convolucionales (CNN) preservan esta estructura bidimensional, utilizando filtros (kernels) que se deslizan sobre la imagen para extraer características manteniendo la noción de localidad espacial, lo cual es vital para localizar objetos dentro del cuadro.

In [2]:
# Leer el dataset y mostrar la estructura de las etiquetas
csv_path = 'muestra_300-20260630T014845Z-3-001/muestra_300/muestra_300.csv'
df = pd.read_csv(csv_path)

print("Primeras filas del dataset:")
print(df.head(3))

print("\nEjemplo de un Target (longitud variable con múltiples cajas):")
print(df['Target'].iloc[2]) # Mostramos una fila que contiene varias coordenadas separadas por ;

Primeras filas del dataset:
                  Id                                             Target
0  v_cec6zlvav9_0038                                               none
1  v_1us0n1cwou_0010  1 35.4 474.725 70.8 33.83 0;1 84.22 420.235 83...
2  v_fw861t0iss_0048                                               none

Ejemplo de un Target (longitud variable con múltiples cajas):
none


## Incompatibilidad Dimensional

Las librerías como `scikit-learn` están diseñadas asumiendo que la matriz de etiquetas (target `y`) tiene una forma estática y uniforme. Se espera un vector unidimensional para tareas de clasificación estándar o una matriz de tamaño fijo para regresión multivariada.

Esto choca con la naturaleza de la **Detección de Objetos**, donde la cantidad de tensores de salida (las predicciones de bounding boxes) varía dinámicamente según lo que haya en la imagen. `scikit-learn` no soporta pasar etiquetas `y` de longitud variable (ragged tensors).

In [3]:
# Simulación del error por dimensionalidad dinámica
clf = MLPClassifier(hidden_layer_sizes=(100,))

# Simulamos X (3 imágenes aplanadas ficticias de 100 features)
X_dummy = np.random.rand(3, 100)

# Extraemos 3 targets reales y los convertimos a listas numéricas de longitudes variables
y_list = []
for target in df['Target'].iloc[2:5].values:
    if target == 'none':
        y_list.append([])
    else:
        coords = []
        for box in target.split(';'):
            coords.extend([float(x) for x in box.split(' ')])
        y_list.append(coords)

try:
    print("Intentando entrenar con etiquetas de longitudes variables:", [len(y) for y in y_list])
    clf.fit(X_dummy, y_list)
except ValueError as e:
    print("\n--- ERROR CAPTURADO ---")
    print("ValueError:", e)
    print("--- FIN DEL ERROR ---")
    print("\nConclusión: El método fit falla estrepitosamente debido a la incompatibilidad de dimensiones (vectores de tamaño variable).")

Intentando entrenar con etiquetas de longitudes variables: [0, 84, 156]

--- ERROR CAPTURADO ---
ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.
--- FIN DEL ERROR ---

Conclusión: El método fit falla estrepitosamente debido a la incompatibilidad de dimensiones (vectores de tamaño variable).
